<a href="https://colab.research.google.com/github/zhouning/alphaearth-training-system/blob/master/colab/train_loveda_crossdomain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LoveDA Cross-Domain PEFT Replication (paper12 Section 9.4)

Runs 5 PEFT methods × 2 directions (U→R, R→U) × 3 seeds = 30 runs on Colab Pro L4.
Resume-aware: rerunning a partially-complete cell skips finished `(method, seed)` rows.

Drive layout (assumed):
- `MyDrive/Prithvi_100M.pt`        — backbone weights (already present from paper58)
- `MyDrive/loveda/raw/`            — torchgeo download cache (~3 GB, one-time)
- `MyDrive/loveda/runs/u2r/`       — checkpoints + per-epoch state for U→R
- `MyDrive/loveda/runs/r2u/`       — checkpoints + per-epoch state for R→U
- `MyDrive/loveda/results/`        — merged JSON (final paper artifact)

Wall-clock budget: ~12.5 h per direction on L4. Colab Pro session limit is 12 h, so plan for at least one disconnect per direction. The `--checkpoint-dir` flag enables resume.

Reversal protocol (spec §5): if Houlsby ≈ Linear or LoRA ≥ +0.05 vs Linear, **stop and re-evaluate** before paper integration.

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 2. GPU + Python sanity check
!nvidia-smi
!python --version

In [ ]:
# 3. Clone the repo (public, no auth needed)
%cd /content
!rm -rf AlphaEarth-System
!git clone https://github.com/zhouning/alphaearth-training-system.git AlphaEarth-System
%cd AlphaEarth-System
!git log --oneline -5

In [ ]:
# 4. Install geoadapter (editable) + torchgeo + pyyaml
!pip install -q -e . torchgeo pyyaml

In [ ]:
# 5. Stage Prithvi backbone weights into the path geoadapter expects
import shutil, os
os.makedirs("data/weights/prithvi", exist_ok=True)
DRIVE_WEIGHTS = "/content/drive/MyDrive/Prithvi_100M.pt"
DST = "data/weights/prithvi/Prithvi_100M.pt"
if not os.path.exists(DST):
    shutil.copy(DRIVE_WEIGHTS, DST)
print("Prithvi weights:", os.path.getsize(DST), "bytes")

In [ ]:
# 6. Symlink LoveDA cache to Drive so the ~3 GB download survives session resets
import os
os.makedirs("/content/drive/MyDrive/loveda/raw", exist_ok=True)
os.makedirs("data/weights/raw_data", exist_ok=True)
if not os.path.islink("data/weights/raw_data/loveda"):
    os.symlink("/content/drive/MyDrive/loveda/raw", "data/weights/raw_data/loveda")
print("LoveDA root:", os.path.realpath("data/weights/raw_data/loveda"))

In [ ]:
# 7. Trigger torchgeo download (first run only; subsequent runs reuse the Drive cache)
from geoadapter.data.datasets import load_loveda
ds_smoke = load_loveda(root="data/weights/raw_data/loveda", domain="urban", split="train", max_samples=5)
print(f"LoveDA urban-train sample count: {len(ds_smoke)}")
img, mask = ds_smoke[0]
print(f"image shape={tuple(img.shape)}, mask shape={tuple(mask.shape)}, mask classes={set(mask.unique().tolist())}")

In [ ]:
# 8. PHASE 2 — U→R direction (15 runs: 5 methods × 3 seeds, 30 epochs each)
# Wall-clock ~12.5 h on L4. If session disconnects, just rerun this cell —
# run_benchmark.py skips already-completed (method, seed) rows via --checkpoint-dir.
!python -m geoadapter.bench.run_benchmark \
    --config geoadapter/bench/configs/loveda_lulc_u2r.yaml \
    --output /content/drive/MyDrive/loveda/results/loveda_lulc_seg_u2r.json \
    --checkpoint-dir /content/drive/MyDrive/loveda/runs/u2r \
    --checkpoint-every 2

In [ ]:
# 9. Verify all 15 U→R rows are present before moving on
import json
rows = json.loads(open("/content/drive/MyDrive/loveda/results/loveda_lulc_seg_u2r.json").read())
assert len(rows) == 15, f"expected 15 rows, got {len(rows)}"
print("U→R completed pairs:", sorted({(r['method'], r['seed']) for r in rows}))

In [ ]:
# 10. PHASE 2 — R→U direction (15 runs)
# Same wall-clock and resume properties as U→R.
!python -m geoadapter.bench.run_benchmark \
    --config geoadapter/bench/configs/loveda_lulc_r2u.yaml \
    --output /content/drive/MyDrive/loveda/results/loveda_lulc_seg_r2u.json \
    --checkpoint-dir /content/drive/MyDrive/loveda/runs/r2u \
    --checkpoint-every 2

In [ ]:
# 11. Verify all 15 R→U rows + merge into the canonical paper artifact
import json
rows = json.loads(open("/content/drive/MyDrive/loveda/results/loveda_lulc_seg_r2u.json").read())
assert len(rows) == 15, f"expected 15 rows, got {len(rows)}"
print("R→U completed pairs:", sorted({(r['method'], r['seed']) for r in rows}))

!python -m geoadapter.bench.run_loveda_crossdomain \
    --u2r-config geoadapter/bench/configs/loveda_lulc_u2r.yaml \
    --r2u-config geoadapter/bench/configs/loveda_lulc_r2u.yaml \
    --output /content/drive/MyDrive/loveda/results/loveda_lulc_seg.json \
    --skip-runs

import os, shutil
os.makedirs("results/loveda", exist_ok=True)
shutil.copy("/content/drive/MyDrive/loveda/results/loveda_lulc_seg.json",
            "results/loveda/loveda_lulc_seg.json")
print("Local copy ready at results/loveda/loveda_lulc_seg.json")

In [ ]:
# 12. Summary table for LaTeX + reversal-protocol gate (spec §5)
import json, statistics
rows = json.loads(open("results/loveda/loveda_lulc_seg.json").read())
agg = {}
for r in rows:
    agg.setdefault((r['direction'], r['method']), []).append(r['mIoU'])
print(f"{'direction':<8} {'method':<14} {'mean':>8} {'std':>8} {'n':>3}")
for (d, m), v in sorted(agg.items()):
    print(f"{d:<8} {m:<14} {statistics.mean(v):>8.4f} {statistics.stdev(v) if len(v)>1 else 0:>8.4f} {len(v):>3}")

# Reversal protocol: Houlsby − Linear must be ≥ +0.05 on at least one direction,
# AND |LoRA − Linear| < 0.05 on both directions. Otherwise STOP, do not write the paper.
def mean_for(d, m): return statistics.mean(agg[(d, m)])
u2r_houlsby_gain = mean_for('U->R', 'houlsby')      - mean_for('U->R', 'linear_probe')
r2u_houlsby_gain = mean_for('R->U', 'houlsby')      - mean_for('R->U', 'linear_probe')
u2r_lora_gap    = abs(mean_for('U->R', 'lora_r8')   - mean_for('U->R', 'linear_probe'))
r2u_lora_gap    = abs(mean_for('R->U', 'lora_r8')   - mean_for('R->U', 'linear_probe'))
print()
print(f"Houlsby − Linear   U→R: {u2r_houlsby_gain:+.4f}   R→U: {r2u_houlsby_gain:+.4f}")
print(f"|LoRA − Linear|    U→R: {u2r_lora_gap:.4f}   R→U: {r2u_lora_gap:.4f}")
houlsby_ok = (u2r_houlsby_gain >= 0.05) or (r2u_houlsby_gain >= 0.05)
lora_ok    = (u2r_lora_gap < 0.05) and (r2u_lora_gap < 0.05)
if houlsby_ok and lora_ok:
    print("\nREVERSAL PROTOCOL: PASS — ranking reproduces, proceed to Phase 3 paper integration.")
else:
    print("\nREVERSAL PROTOCOL: FAIL — STOP. Do not write Section 9.4. Re-evaluate per spec §5.")